# Exercise 1 — Decision Trees and Intrinsic Interpretability

**PCS956 • Module C • Week 1**

## Research question

**What can we understand directly from a small decision tree, and where do the limits of intrinsic interpretability appear?**

In this exercise, we will:

- inspect the Titanic dataset and prepare it for modelling;
- train a deliberately small decision tree;
- inspect its complete structure and textual rules;
- trace an exact local decision path;
- examine how leaf probabilities are obtained;
- study tree complexity in relation to predictive performance; and
- examine the stability of the root split under resampling.

No post-hoc explainer is used in this exercise. All explanations come directly from the fitted decision tree.


## 1. Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text


## 2. Load and inspect the dataset

Each row represents one passenger.

The target variable is:

- `survived = 0` — the passenger was recorded as not surviving;
- `survived = 1` — the passenger was recorded as surviving.

The predictors used in this practical are:

| Feature | Meaning |
|---|---|
| `pclass` | Passenger class: 1st, 2nd, or 3rd |
| `sex` | Recorded sex |
| `age` | Age in years |
| `sibsp` | Number of siblings/spouses aboard |
| `parch` | Number of parents/children aboard |
| `fare` | Passenger fare |
| `embarked` | Port of embarkation |

For `embarked`:

- `C` = Cherbourg
- `Q` = Queenstown
- `S` = Southampton

We first inspect the raw data before fitting a model.


In [ ]:
# Load the Titanic dataset.
titanic = sns.load_dataset("titanic")

# Variables used by the model.
FEATURES = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked"
]

# The target is the outcome that the model will predict.
TARGET = "survived"

# Show the shape of the full dataset.
print("Dataset shape:", titanic.shape)

# Display the first five rows of the variables used in this practical.
display(titanic[FEATURES + [TARGET]].head())


### Missing values

Before modelling, we check whether any selected variables contain missing values.


In [ ]:
missing_values = titanic[FEATURES + [TARGET]].isna().sum()

display(
    missing_values.to_frame("missing_count")
)


### Target distribution

A classification model predicts one of the two target classes.

The plot below shows how many passengers belong to each observed outcome class.


In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    data=titanic,
    x="survived"
)

plt.xticks(
    [0, 1],
    ["Did not survive", "Survived"]
)

plt.xlabel("Observed outcome")
plt.ylabel("Number of passengers")
plt.title("Target distribution")

plt.show()


### Descriptive patterns in the dataset

The following summaries show observed survival rates across selected groups.

These are **descriptive associations** in the historical dataset. They do not establish causal effects.


In [ ]:
# Observed survival rate by recorded sex.
survival_by_sex = (
    titanic
    .groupby("sex", observed=False)["survived"]
    .mean()
    .reset_index()
)

display(survival_by_sex)

plt.figure(figsize=(6, 4))

sns.barplot(
    data=survival_by_sex,
    x="sex",
    y="survived"
)

plt.ylim(0, 1)
plt.xlabel("Recorded sex")
plt.ylabel("Observed survival rate")
plt.title("Observed survival rate by sex")

plt.show()


In [ ]:
# Observed survival rate by passenger class.
survival_by_class = (
    titanic
    .groupby("pclass", observed=False)["survived"]
    .mean()
    .reset_index()
)

display(survival_by_class)

plt.figure(figsize=(6, 4))

sns.barplot(
    data=survival_by_class,
    x="pclass",
    y="survived"
)

plt.ylim(0, 1)
plt.xlabel("Passenger class")
plt.ylabel("Observed survival rate")
plt.title("Observed survival rate by passenger class")

plt.show()


### What to notice before modelling

From the exploratory analysis, identify:

- whether the target classes are equally represented;
- which selected predictors contain missing values;
- whether observed survival rates differ across the groups shown.

These observations help us understand the dataset before fitting the model. They are descriptive patterns only and should not be interpreted as causal effects.


## 3. Prepare the data

We now separate the predictors from the target and create training and test sets.

The preprocessing steps are fitted only on the training data and then applied to the test data. This avoids using information from the test set when preparing the model inputs.


In [ ]:
RANDOM_STATE = 42

# X contains the predictor variables.
X = titanic[FEATURES].copy()

# y contains the target that the model will predict.
y = titanic[TARGET].copy()

# Split the data before fitting any preprocessing.
# stratify=y keeps approximately the same target-class proportions
# in both the training and test sets.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training observations:", len(X_train_raw))
print("Test observations:", len(X_test_raw))


### Preprocessing strategy

Numerical and categorical variables require different preprocessing.

For numerical variables:

- missing values are replaced with the median calculated from the training set.

For categorical variables:

- missing values are replaced with the most frequent category in the training set;
- one-hot encoding converts each category into a binary indicator column.

All categories are kept in the encoded representation. For example, `sex` becomes:

- `sex_female`
- `sex_male`

These two columns are redundant from a purely mathematical perspective because one can be inferred from the other. Keeping both does not prevent a decision tree from fitting correctly, but it means equivalent splits may be represented using either encoded column.

Similarly, `embarked` becomes:

- `embarked_C`
- `embarked_Q`
- `embarked_S`


In [ ]:
# Separate numerical and categorical predictors.
numeric_features = [
    "pclass",
    "age",
    "sibsp",
    "parch",
    "fare"
]

categorical_features = [
    "sex",
    "embarked"
]

# Numerical pipeline:
# replace missing values with the median from the training data.
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

# Categorical pipeline:
# replace missing values with the most frequent category,
# then create one binary column for each observed category.
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

# Apply the appropriate preprocessing to each feature group.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ],
    verbose_feature_names_out=False
)

# Learn preprocessing parameters from the training set.
X_train = preprocessor.fit_transform(X_train_raw)

# Apply the already-fitted preprocessing to the test set.
X_test = preprocessor.transform(X_test_raw)

# Retrieve the transformed feature names.
feature_names = preprocessor.get_feature_names_out()

# Convert the transformed arrays to DataFrames.
X_train_df = pd.DataFrame(
    X_train,
    columns=feature_names,
    index=X_train_raw.index
)

X_test_df = pd.DataFrame(
    X_test,
    columns=feature_names,
    index=X_test_raw.index
)

print("Transformed features:")
print(list(feature_names))


### Inspect the transformed data

The model does not receive the original categorical strings directly. It receives the transformed numerical representation shown below.


In [ ]:
display(X_train_df.head())


## 4. Train a small decision tree

We fit a decision tree with `max_depth=3`.

A small maximum depth limits the structural complexity of the tree and makes the complete fitted model easier to inspect.


In [ ]:
# Create the classifier.
tree_small = DecisionTreeClassifier(
    max_depth=3,
    random_state=RANDOM_STATE
)

# Fit the tree on the transformed training data.
tree_small.fit(
    X_train_df,
    y_train
)

# Predict the training and test sets.
train_pred = tree_small.predict(X_train_df)
test_pred = tree_small.predict(X_test_df)

# Report predictive performance.
print(
    "Training accuracy:",
    round(
        accuracy_score(y_train, train_pred),
        3
    )
)

print(
    "Test accuracy:",
    round(
        accuracy_score(y_test, test_pred),
        3
    )
)

print(
    "Test balanced accuracy:",
    round(
        balanced_accuracy_score(
            y_test,
            test_pred
        ),
        3
    )
)

# Report structural properties of the fitted tree.
print(
    "\nTree depth:",
    tree_small.get_depth()
)

print(
    "Number of nodes:",
    tree_small.tree_.node_count
)

print(
    "Number of leaves:",
    tree_small.get_n_leaves()
)


### Interpreting the performance measures

- **Accuracy** is the proportion of observations classified correctly.
- **Balanced accuracy** is the average recall across the target classes and is useful when the classes are not equally represented.

The structural quantities describe the fitted model itself:

- **depth** — the maximum number of successive splits from the root to a leaf;
- **nodes** — all decision and leaf nodes in the tree;
- **leaves** — terminal nodes that produce predictions.

Predictive performance and structural complexity measure different properties of the model.


## 5. Global intrinsic interpretation

A small decision tree can be inspected directly.

The complete tree provides a **global view** of the fitted model because its learned decision structure is visible.


In [ ]:
plt.figure(figsize=(20, 10))

plot_tree(
    tree_small,
    feature_names=feature_names,
    class_names=[
        "did not survive",
        "survived"
    ],
    filled=True,
    rounded=True,
    impurity=True,
    precision=2
)

plt.title("Small decision tree — Titanic")
plt.show()


### How to read a node

A decision node contains a rule such as:

`sex_female <= 0.5`

Because the encoded values are binary:

- `sex_female = 0` follows the left branch;
- `sex_female = 1` follows the right branch.

The same underlying binary partition could also be represented with the complementary column `sex_male`.

Other information shown in a node includes:

- the impurity value;
- the number of training observations reaching that node;
- the class counts;
- the predicted class.

These rules describe the behaviour of the fitted tree. They should not be interpreted as causal relationships.


## 6. Inspect the fitted rules as text

The same tree can be represented as a sequence of textual rules.

These rules are part of the fitted decision tree itself. They are not a post-hoc approximation.


In [ ]:
print(
    export_text(
        tree_small,
        feature_names=list(feature_names),
        decimals=2
    )
)


## 7. Explain one prediction with its exact decision path

The full tree provides a global view.

For a single passenger, we can instead inspect the exact sequence of nodes traversed by that observation. This gives a **local explanation** for the prediction made by this fitted tree.

We select a misclassified test example if one is available. This is useful because it demonstrates that an explanation can be exact with respect to the model even when the prediction itself is wrong.


In [ ]:
# Count how many nodes each test observation passes through.
path_lengths = (
    tree_small
    .decision_path(X_test_df)
    .sum(axis=1)
    .A1
)

# Find test observations that were misclassified.
misclassified_positions = np.flatnonzero(
    test_pred != y_test.to_numpy()
)

if len(misclassified_positions) > 0:
    # Among misclassified cases, choose one with a relatively long path.
    sample_position = int(
        misclassified_positions[
            np.argmax(
                path_lengths[
                    misclassified_positions
                ]
            )
        ]
    )
    selection_reason = "misclassified test case"
else:
    # If no misclassified case exists, choose the deepest available path.
    sample_position = int(
        np.argmax(path_lengths)
    )
    selection_reason = "deepest available test path"

# Raw values are useful for human interpretation.
raw_sample = X_test_raw.iloc[[sample_position]]

# Transformed values are the representation seen by the tree.
sample = X_test_df.iloc[[sample_position]]

true_class = int(
    y_test.iloc[sample_position]
)

predicted_class = int(
    tree_small.predict(sample)[0]
)

probabilities = tree_small.predict_proba(sample)[0]

print("Selected:", selection_reason)

display(raw_sample)

print(
    "Observed outcome:",
    "survived"
    if true_class == 1
    else "did not survive"
)

print(
    "Predicted outcome:",
    "survived"
    if predicted_class == 1
    else "did not survive"
)

print(
    "Predicted probabilities:"
)

print(
    "  did not survive:",
    round(probabilities[0], 3)
)

print(
    "  survived:",
    round(probabilities[1], 3)
)


In [ ]:
# Retrieve the nodes visited by the selected passenger.
node_indicator = tree_small.decision_path(sample)

# Identify the terminal leaf.
leaf_id = tree_small.apply(sample)[0]

# Access the fitted tree structure.
tree = tree_small.tree_

print("Exact decision path:\n")

for node_id in node_indicator.indices:
    if node_id == leaf_id:
        print(
            f"Node {node_id}: LEAF → prediction = "
            f"{'survived' if predicted_class == 1 else 'did not survive'}"
        )
    else:
        # Feature used at this node.
        feature_index = tree.feature[node_id]
        feature_name = feature_names[feature_index]

        # Threshold used by the split.
        threshold = tree.threshold[node_id]

        # Selected passenger's value for this feature.
        feature_value = sample.iloc[
            0,
            feature_index
        ]

        # Determine which side of the split is followed.
        relation = (
            "<="
            if feature_value <= threshold
            else ">"
        )

        print(
            f"Node {node_id}: "
            f"{feature_name} = {feature_value:.3f} "
            f"{relation} {threshold:.3f}"
        )


### Interpretation

The decision path is an exact description of the sequence of rules used by this fitted tree for this prediction.

If the prediction is wrong, the explanation can still be faithful to the model.

This distinction is important:

> **An exact explanation of model behaviour does not imply that the model prediction is correct.**


## 8. Where do the class probabilities come from?

For a classification tree, `predict_proba()` is based on the class proportions among training observations that reached the same leaf.

We can inspect those training observations directly.


In [ ]:
# Leaf reached by the selected test passenger.
leaf_node = tree_small.apply(sample)[0]

# Leaf reached by every training observation.
train_leaf_ids = tree_small.apply(X_train_df)

# Training observations in the same leaf.
same_leaf = (
    train_leaf_ids == leaf_node
)

leaf_targets = y_train.to_numpy()[
    same_leaf
]

# Count each class in the leaf.
count_0 = np.sum(
    leaf_targets == 0
)

count_1 = np.sum(
    leaf_targets == 1
)

total = len(leaf_targets)

leaf_table = pd.DataFrame({
    "class": [
        "did not survive",
        "survived"
    ],
    "training_count_in_leaf": [
        count_0,
        count_1
    ],
    "leaf_proportion": [
        count_0 / total,
        count_1 / total
    ]
})

display(leaf_table)

print(
    "Model probabilities:",
    np.round(
        probabilities,
        3
    )
)


The class proportions in the leaf explain where the tree's predicted probabilities come from.

This transparency does not, by itself, establish that the probabilities are well calibrated for new populations or future data.


## 9. Complexity experiment

A decision tree belongs to an interpretable model family, but a large fitted tree may still be difficult to inspect.

We therefore fit trees with different maximum depths and compare:

- training accuracy;
- test accuracy;
- balanced accuracy;
- number of nodes;
- number of leaves.

This experiment shows how predictive performance and structural complexity can change together for this dataset and model configuration.


In [ ]:
depth_values = [
    1,
    2,
    3,
    4,
    5,
    6,
    8,
    None
]

records = []

for max_depth in depth_values:
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        random_state=RANDOM_STATE
    )

    model.fit(
        X_train_df,
        y_train
    )

    train_predictions = model.predict(
        X_train_df
    )

    test_predictions = model.predict(
        X_test_df
    )

    records.append({
        "max_depth": (
            "None"
            if max_depth is None
            else str(max_depth)
        ),
        "fitted_depth": model.get_depth(),
        "nodes": model.tree_.node_count,
        "leaves": model.get_n_leaves(),
        "train_accuracy": accuracy_score(
            y_train,
            train_predictions
        ),
        "test_accuracy": accuracy_score(
            y_test,
            test_predictions
        ),
        "test_balanced_accuracy":
            balanced_accuracy_score(
                y_test,
                test_predictions
            )
    })

complexity_results = pd.DataFrame(
    records
)

display(complexity_results)


In [ ]:
x = np.arange(
    len(complexity_results)
)

plt.figure(figsize=(9, 5))

plt.plot(
    x,
    complexity_results["train_accuracy"],
    marker="o",
    label="Training accuracy"
)

plt.plot(
    x,
    complexity_results["test_accuracy"],
    marker="o",
    label="Test accuracy"
)

plt.xticks(
    x,
    complexity_results["max_depth"]
)

plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title(
    "Predictive performance as tree complexity changes"
)

plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    x,
    complexity_results["nodes"],
    marker="o"
)

plt.xticks(
    x,
    complexity_results["max_depth"]
)

plt.xlabel("max_depth")
plt.ylabel("Number of nodes")
plt.title(
    "Structural complexity as tree depth changes"
)

plt.show()


### Interpretation

As the tree becomes more flexible:

- training accuracy will often increase;
- test performance may improve, remain similar, or decrease;
- the number of nodes can grow rapidly;
- the complete model becomes more difficult to inspect.

The results from one dataset do not establish a universal trade-off between predictive performance and interpretability.


## 10. Stability experiment

Interpretability and stability are different properties.

A model may be easy to inspect but still change substantially when trained on a slightly different sample.

We investigate the **root split** by repeatedly:

1. drawing a bootstrap sample from the training data;
2. fitting another depth-3 tree;
3. recording the feature selected at the root.

Because `sex_female` and `sex_male` are complementary encodings of the same original variable, we report both:

- the original feature concept (`sex`);
- the exact encoded column selected by the fitted tree.


In [ ]:
rng = np.random.default_rng(
    RANDOM_STATE
)

stability_records = []

for run in range(30):
    # Sample training-row positions with replacement.
    bootstrap_positions = rng.integers(
        0,
        len(X_train_df),
        size=len(X_train_df)
    )

    X_boot = X_train_df.iloc[
        bootstrap_positions
    ]

    y_boot = y_train.iloc[
        bootstrap_positions
    ]

    # Fit another depth-3 tree.
    model = DecisionTreeClassifier(
        max_depth=3,
        random_state=run
    )

    model.fit(
        X_boot,
        y_boot
    )

    # Identify the encoded feature used at the root.
    root_feature_index = (
        model.tree_.feature[0]
    )

    if root_feature_index >= 0:
        root_feature_encoded = (
            feature_names[
                root_feature_index
            ]
        )

        root_threshold = (
            model.tree_.threshold[0]
        )
    else:
        root_feature_encoded = "no split"
        root_threshold = np.nan

    # Group complementary sex indicators back to the
    # original feature concept.
    if root_feature_encoded.startswith(
        "sex_"
    ):
        root_feature_concept = "sex"
    else:
        root_feature_concept = (
            root_feature_encoded
        )

    stability_records.append({
        "run": run,
        "root_feature_encoded":
            root_feature_encoded,
        "root_feature_concept":
            root_feature_concept,
        "root_threshold":
            root_threshold
    })

stability_results = pd.DataFrame(
    stability_records
)

# Summary by original feature concept.
root_summary = (
    stability_results[
        "root_feature_concept"
    ]
    .value_counts()
    .rename_axis(
        "root_feature"
    )
    .reset_index(
        name="count"
    )
)

display(root_summary)

plt.figure(figsize=(9, 4))

plt.bar(
    root_summary["root_feature"],
    root_summary["count"]
)

plt.ylabel(
    "Number of bootstrap fits"
)

plt.title(
    "Root feature selected across bootstrap samples"
)

plt.xticks(
    rotation=35,
    ha="right"
)

plt.show()

# Exact encoded columns selected at the root.
encoded_root_summary = (
    stability_results[
        "root_feature_encoded"
    ]
    .value_counts()
    .rename_axis(
        "encoded_root_feature"
    )
    .reset_index(
        name="count"
    )
)

print(
    "Exact encoded root columns:"
)

display(
    encoded_root_summary
)


### Interpretation

If the same original feature concept appears in most bootstrap fits, the root variable is relatively stable under this resampling procedure.

If several different features appear, the root split is more sample-sensitive.

This experiment evaluates only one aspect of structural stability. It does not show that the entire tree is stable, and it does not establish causal importance.


## 11. Student task

Now repeat part of the analysis independently.

### Task

Fit two decision trees using:

- `max_depth=2`
- `max_depth=6`

For each model:

1. report training accuracy;
2. report test accuracy;
3. report test balanced accuracy;
4. report the number of nodes and leaves;
5. inspect the plotted tree;
6. identify the root feature;
7. decide which of the two trees is easier to inspect globally.

Then answer:

- Does the deeper tree improve predictive performance on this test split?
- How much does structural complexity increase?
- Is the more complex tree necessarily the better model?
- What can you conclude from this experiment, and what can you **not** conclude?

> Keep the interpretation specific to these fitted models and this dataset. Do not infer a universal accuracy–interpretability trade-off from two trees.


## 12. Questions for discussion

1. What information can you obtain directly from the complete tree?
2. Why is the full tree a global interpretation?
3. Why is one decision path a local interpretation?
4. Can an exact decision-path explanation accompany an incorrect prediction?
5. How does increasing tree depth affect structural complexity?
6. Does higher complexity always improve test performance?
7. Why can `sex_female` and `sex_male` represent equivalent splits?
8. What does the bootstrap experiment tell us about the root split?
9. What does the bootstrap experiment **not** tell us?
10. Why should the tree rules not be interpreted causally?


## 13. Takeaway

A small decision tree can provide intrinsic interpretability because relevant parts of its fitted structure can be inspected directly.

In this exercise:

- the complete tree provides a **global view** of the fitted decision structure;
- the decision path provides an **exact local explanation** for one prediction;
- leaf class proportions show how the tree obtains its predicted class probabilities;
- increasing structural complexity can make the full model harder to inspect;
- predictive performance and interpretability are related to the fitted model and problem, not by a universal rule;
- stability is a separate property and can be investigated by refitting the model under resampling.

The explanations in this notebook describe the behaviour of the fitted model. They do not, by themselves, establish causality, correctness, fairness, robustness, or suitability for deployment.
